# Week 3: Unsupervised Learning and Clustering Analysis
## Laptop Dataset — K-Means Clustering

**Objective:** Apply unsupervised learning to segment laptops into meaningful groups based on hardware specifications, display characteristics, portability, and price.

This notebook uses the cleaned dataset prepared in Week 1 and follows the exploratory analysis performed in Week 2.


## 1. Import Required Libraries


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA

sns.set_theme(style="whitegrid")


## 2. Load the Cleaned Dataset

Update the file path if your notebook is stored in a different folder.


In [ ]:
df = pd.read_csv("../data/cleaned/laptop_cleaned_week1.csv")

df.head()


## 3. Initial Dataset Exploration


In [ ]:
print("Dataset shape:", df.shape)
print("\nColumn names:")
print(df.columns.tolist())

print("\nData types:")
print(df.dtypes)

print("\nMissing values:")
print(df.isnull().sum())


## 4. Select Features for Clustering

The selected numerical features represent:
- **RAM and CPU speed:** computing capability
- **Inches and weight:** physical design and portability
- **Screen width and height:** display resolution
- **Price:** market value

Using multiple dimensions allows clustering to identify broader laptop profiles.


In [ ]:
features = [
    "Ram_GB",
    "Inches",
    "Weight_kg",
    "CPU_GHz",
    "ScreenWidth_px",
    "ScreenHeight_px",
    "Price"
]

X = df[features].apply(pd.to_numeric, errors="coerce").dropna()

print("Shape of clustering data:", X.shape)
X.head()


## 5. Feature Scaling

K-Means uses distance calculations. Since Price and screen resolution have much larger numerical ranges than RAM or CPU speed, the data must be standardized.


In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("Scaled data shape:", X_scaled.shape)
print("Feature means after scaling:")
print(np.round(X_scaled.mean(axis=0), 3))


## 6. Elbow Method

The Elbow Method measures how within-cluster inertia changes as the number of clusters increases.


In [ ]:
inertia = []
k_values = range(2, 9)

for k in k_values:
    model = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=20
    )
    model.fit(X_scaled)
    inertia.append(model.inertia_)

plt.figure(figsize=(8, 5))
plt.plot(k_values, inertia, marker="o")
plt.title("Elbow Method for Selecting Number of Clusters")
plt.xlabel("Number of Clusters (K)")
plt.ylabel("Inertia")
plt.xticks(list(k_values))
plt.tight_layout()
plt.show()


### Interpretation

Look for the point where the reduction in inertia begins to slow down. This point provides visual evidence for a reasonable number of clusters.


## 7. Silhouette Score Analysis

The silhouette score evaluates:
- **Cohesion:** how similar observations are within the same cluster.
- **Separation:** how different a cluster is from other clusters.

Higher scores generally indicate better-defined clusters.


In [ ]:
silhouette_scores = []

for k in k_values:
    model = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=20
    )

    labels = model.fit_predict(X_scaled)
    score = silhouette_score(X_scaled, labels)

    silhouette_scores.append(score)
    print(f"K = {k}: Silhouette Score = {score:.4f}")


In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(k_values, silhouette_scores, marker="o")
plt.title("Silhouette Score for Different Cluster Counts")
plt.xlabel("Number of Clusters (K)")
plt.ylabel("Silhouette Score")
plt.xticks(list(k_values))
plt.tight_layout()
plt.show()

optimal_k = list(k_values)[np.argmax(silhouette_scores)]

print("Suggested optimal K based on silhouette score:", optimal_k)


## 8. Final K-Means Clustering

The final model uses the K value suggested by the silhouette analysis. The Elbow Method should also be considered when interpreting the final choice.


In [ ]:
kmeans = KMeans(
    n_clusters=optimal_k,
    random_state=42,
    n_init=20
)

clusters = kmeans.fit_predict(X_scaled)

df_clustered = X.copy()
df_clustered["Cluster"] = clusters

print(df_clustered["Cluster"].value_counts().sort_index())


## 9. Cluster Visualization: RAM vs Price


In [ ]:
plt.figure(figsize=(9, 6))

sns.scatterplot(
    data=df_clustered,
    x="Ram_GB",
    y="Price",
    hue="Cluster",
    palette="viridis",
    alpha=0.7
)

plt.title("Laptop Clusters: RAM vs Price")
plt.xlabel("RAM (GB)")
plt.ylabel("Price")
plt.legend(title="Cluster")
plt.tight_layout()
plt.show()


### Observation

This plot shows how laptop memory and price contribute to the clustering structure. Some overlap is expected because K-Means uses all selected features, while this chart displays only two.


## 10. Cluster Visualization: CPU Speed vs Price


In [ ]:
plt.figure(figsize=(9, 6))

sns.scatterplot(
    data=df_clustered,
    x="CPU_GHz",
    y="Price",
    hue="Cluster",
    palette="viridis",
    alpha=0.7
)

plt.title("Laptop Clusters: CPU Speed vs Price")
plt.xlabel("CPU Speed (GHz)")
plt.ylabel("Price")
plt.legend(title="Cluster")
plt.tight_layout()
plt.show()


### Interpretation

This visualization provides another view of the relationship between technical capability and price. CPU speed alone does not determine cluster membership because the algorithm also considers memory, weight, screen size, resolution, and price.


## 11. PCA Visualization

PCA reduces the seven-dimensional standardized data into two principal components for visualization. The clustering model still uses all seven standardized features.


In [ ]:
pca = PCA(n_components=2)

X_pca = pca.fit_transform(X_scaled)

pca_df = pd.DataFrame(
    X_pca,
    columns=["PC1", "PC2"]
)

pca_df["Cluster"] = clusters

print(
    "Variance explained by first two components:",
    round(pca.explained_variance_ratio_.sum() * 100, 2),
    "%"
)


In [ ]:
plt.figure(figsize=(9, 6))

sns.scatterplot(
    data=pca_df,
    x="PC1",
    y="PC2",
    hue="Cluster",
    palette="viridis",
    alpha=0.7
)

plt.title("Laptop Clusters Using PCA")
plt.xlabel("Principal Component 1")
plt.ylabel("Principal Component 2")
plt.legend(title="Cluster")
plt.tight_layout()
plt.show()


## 12. Detailed Cluster Profiles

The mean values of each feature are calculated for every cluster. This is the main evidence used to interpret the characteristics of each segment.


In [ ]:
cluster_summary = (
    df_clustered
    .groupby("Cluster")[features]
    .mean()
    .round(2)
)

cluster_summary


In [ ]:
overall_average = X.mean()

comparison = cluster_summary.copy()

for column in features:
    comparison[column] = (
        (cluster_summary[column] - overall_average[column])
        / overall_average[column]
        * 100
    ).round(1)

comparison


The percentage comparison shows whether each cluster is above or below the overall average for each feature.


## 13. Cluster Profile Visualization


In [ ]:
profile = (
    cluster_summary - X.mean()
) / X.std()

plt.figure(figsize=(11, 6))

sns.heatmap(
    profile,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    center=0
)

plt.title("Standardized Cluster Profiles")
plt.xlabel("Features")
plt.ylabel("Cluster")
plt.tight_layout()
plt.show()


### Interpretation

Positive values indicate that the cluster average is above the overall dataset average, while negative values indicate below-average values. This makes it easier to compare clusters across variables measured in different units.


## 14. Cluster Size Analysis


In [ ]:
cluster_counts = df_clustered["Cluster"].value_counts().sort_index()

plt.figure(figsize=(8, 5))

sns.barplot(
    x=cluster_counts.index.astype(str),
    y=cluster_counts.values
)

plt.title("Number of Laptops in Each Cluster")
plt.xlabel("Cluster")
plt.ylabel("Number of Laptops")
plt.tight_layout()
plt.show()

cluster_counts


## 15. Automatic Cluster Characteristics Summary


In [ ]:
for cluster in sorted(cluster_summary.index):

    print("=" * 60)
    print(f"CLUSTER {cluster}")
    print("=" * 60)

    high_features = profile.loc[cluster].sort_values(
        ascending=False
    ).head(3)

    low_features = profile.loc[cluster].sort_values().head(2)

    print("Relatively high characteristics:")
    print(", ".join(high_features.index))

    print("\nRelatively low characteristics:")
    print(", ".join(low_features.index))

    print("\nAverage Price:", round(
        cluster_summary.loc[cluster, "Price"], 2
    ))

    print("Average RAM:", round(
        cluster_summary.loc[cluster, "Ram_GB"], 2
    ), "GB")

    print("Average CPU Speed:", round(
        cluster_summary.loc[cluster, "CPU_GHz"], 2
    ), "GHz")

    print()


## 16. Suggested Interpretation of the Segments

Use the actual output above to describe the clusters.

Possible descriptions include:

- **Entry-Level Segment:** relatively lower specifications and price.
- **Balanced / Mid-Range Segment:** moderate values across most characteristics.
- **Performance-Oriented Segment:** above-average RAM and CPU characteristics.
- **Premium Segment:** relatively high specifications, display characteristics, and price.

Do not assign these names before examining the cluster summary.


## 17. Business Implications

### Inventory Management
Retailers can use the clusters to understand how their inventory is distributed across different specification profiles.

### Pricing Analysis
Products with similar technical profiles but different prices can be investigated further to understand possible brand, design, or feature effects.

### Product Portfolio Analysis
The clusters can help identify whether manufacturers or retailers have gaps in particular specification segments.

### Customer-Oriented Segmentation
The groups may support broad recommendations for users seeking entry-level, balanced, performance-focused, or premium laptops.


## 18. Limitations

1. K-Means is sensitive to feature scaling.
2. The selected number of clusters influences the final segmentation.
3. The analysis uses selected numerical variables and does not directly include all categorical information.
4. PCA is only a two-dimensional visualization of a higher-dimensional structure.
5. Clusters should be interpreted as data-driven profiles, not absolute market categories.


## 19. Conclusion

This Week 3 analysis applies K-Means clustering to the cleaned laptop dataset. Relevant numerical features were selected and standardized to ensure that no variable dominated the distance calculation due to its measurement scale. The Elbow Method and silhouette scores were used to evaluate candidate numbers of clusters. The final clustering model was visualized using RAM, CPU speed, price, and PCA projections.

Detailed cluster profiles were then calculated to identify the relative strengths and characteristics of each segment. The resulting groups can support product segmentation, inventory analysis, pricing research, and further machine-learning experimentation.
